# Python & PySpark Interview Notes

---

# 1. Python Decorators

## What is a Decorator?

A decorator is a function that adds extra functionality to another function without modifying its original code.

### Real-Life Example

Think of a decorator as a security guard at an office entrance.

- Before entering → Identity verification
- After leaving → Exit logging

The employee does their work normally, but extra actions happen before and after.

---

## Example

```python
def decorator(func):

    def wrapper():
        print("Before Function")
        func()
        print("After Function")

    return wrapper


@decorator
def greet():
    print("Hello")

greet()
```

### Output

```python
Before Function
Hello
After Function
```

---

## Common Use Cases

- Logging
- Authentication
- Authorization
- Exception Handling
- Performance Monitoring

---

## Interview Answer

**A decorator is a function that wraps another function to extend or modify its behavior without changing the original function's source code. Decorators are commonly used for logging, authentication, validation, and monitoring.**

---

# 2. Python Generators

## What is a Generator?

A generator is a special type of function that returns values one at a time using the `yield` keyword.

Instead of storing all values in memory, generators produce values on demand.

---

## Why Use Generators?

### Normal List

```python
numbers = [i for i in range(1000000)]
```

- Stores all values in memory.

### Generator

```python
def numbers():
    for i in range(1000000):
        yield i
```

- Generates values one by one.
- Memory efficient.

---

## Example

```python
def numbers():
    for i in range(5):
        yield i

g = numbers()

print(next(g))
print(next(g))
print(next(g))
```

### Output

```python
0
1
2
```

---

## yield vs return

### return

```python
def test():
    return 10
```

Function ends immediately.

### yield

```python
def test():
    yield 10
    yield 20
```

Produces values one at a time.

---

## Interview Answer

**A generator is a memory-efficient function that produces values lazily using the `yield` keyword instead of returning all values at once.**

---

# 3. Cache vs Persist

## Why Do We Need Them?

Without caching, Spark recomputes the entire DataFrame every time it is used.

```python
df.count()

df.show()

df.collect()
```

Spark performs the same computations again and again.

---

# Cache

```python
df.cache()
```

Stores the DataFrame in memory.

Default Storage Level:

```python
MEMORY_ONLY
```

---

# Persist

```python
from pyspark import StorageLevel

df.persist(StorageLevel.MEMORY_AND_DISK)
```

Allows custom storage options.

---

## Storage Levels

```python
MEMORY_ONLY

MEMORY_AND_DISK

DISK_ONLY

MEMORY_AND_DISK_SER
```

---

## Difference Between Cache and Persist

### Cache

```python
df.cache()
```

- Shortcut for persist
- Default = MEMORY_ONLY

### Persist

```python
df.persist(StorageLevel.MEMORY_AND_DISK)
```

- Custom storage level
- More flexible

---

## Interview Answer

**Cache is a shorthand for persist **th the default storage level of MEMORY_ONLY. Persist provides more flexibility by allowing different storage levels such as MEMORY_AND_DISK and DISK_ONLY.**

---

# 4. Catalyst Optimizer

## What is Catalyst Optimizer?

Catalyst Optimizer is the query optimization engine of Spark SQL.

It analyzes and optimizes queries before execution.

---

## Real-Life Example

Think of Catalyst as Google Maps.

You provide the destination.

Google Maps finds the best route.

Similarly, Catalyst finds the most efficient execution plan.

---

## Optimizations Performed

### Predicate Pushdown

```python
df.filter(df.salary > 50000)
```

Filter data as early as possible.

---

### Column Pruning

```python
df.select("name", "salary")
```

Read only required columns.

---

### Join Optimization

Chooses the most efficient join strategy.

---

## Interview Answer

**Catalyst Optimizer is Spark SQL's optimization engine that improves query performance by applying techniques such as predicate pushdown, column pruning, and join optimization before execution.**

---

# 5. Tungsten Execution Engine

## What is Tungsten?

Tungsten is Spark's physical execution engine.

It focuses on efficient memory management and CPU optimization.

---

## Relationship with Catalyst

```text
Catalyst -> Creates Optimized Plan

Tungsten -> Executes Optimized Plan
```

---

## Benefits

- Better Memory Management
- Lower Garbage Collection Overhead
- Faster CPU Processing
- Improved Performance

---

## Interview Answer

**Tungsten is Spark's execution engine that improves performance through optimized memory management, code generation, and efficient CPU utilization.**

---

# 6. Adaptive Query Execution (AQE)

## What is AQE?

AQE stands for Adaptive Query Execution.

It allows Spark to modify the execution plan at runtime based on actual data statistics.

---

## Why AQE?

Traditional Spark:

```text
Plan Created
Plan Executed
```

No changes allowed.

---

With AQE:

```text
Plan Created

Runtime Statistics Collected

Plan Optimized Again

Execution Continues
```

---

## Benefits

### Dynamic Join Strategy

Convert normal join into broadcast join.

### Better Partition Management

Reduce unnecessary partitions.

### Handle Data Skew

Optimize skewed joins automatically.

---

## Enable AQE

```python
spark.conf.set(
    "spark.sql.adaptive.enabled",
    "true"
)
```

---

## Interview Answer

**Adaptive Query Execution (AQE) allows Spark to optimize execution plans dynamically during runtime using actual data statistics. It improves performance by optimizing joins, partitions, and skewed data handling.**

---

# 7. Data Skew

## What is Data Skew?

Data skew occurs when some partitions contain significantly more data than others.

---

## Example

```text
Partition 1 -> 100 Rows

Partition 2 -> 200 Rows

Partition 3 -> 10 Million Rows
```

Partition 3 becomes a bottleneck.

---

## Problems Caused

- Slow execution
- Long-running tasks
- Uneven resource utilization

---

## Interview Answer

**Data skew occurs when data is unevenly distributed across partitions, causing some executors to process significantly more data than others and resulting in performance issues.**

---

# 8. Salting

## What is Salting?

Salting is a technique used to solve data skew problems.

Random values are added to heavily skewed keys, distributing data across multiple partitions.

---

## Before Salting

```text
India -> 1,000,000 Rows
US    -> 100 Rows
```

Most data goes to a single partition.

---

## After Salting

```text
India_1
India_2
India_3
India_4
India_5
```

Data becomes evenly distributed.

---

## Example

```python
from pyspark.sql.functions import rand

df = df.withColumn(
    "salt",
    (rand() * 5).cast("int")
)
```

---

## Interview Answer

**Salting is a technique used to reduce data skew by adding random values to skewed keys so that records are distributed more evenly across partitions.**

---

# 9. Window Functions

## What is a Window Function?

A window function performs calculations across a group of rows while retaining all individual rows.

Unlike `groupBy()`, it does not collapse rows.

---

## Example Data

```text
Name   Dept   Salary

A      IT     50000
B      IT     60000
C      IT     70000
D      HR     40000
E      HR     45000
```

---

# row_number()

Assigns a unique sequential number.

```python
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy("Dept") \
                    .orderBy("Salary")

df.withColumn(
    "row_num",
    row_number().over(window_spec)
)
```

---

# rank()

Duplicate values receive the same rank.

Gaps will appear.

### Example

```text
Salary

100
100
90
```

Output:

```text
1
1
3
```

---

# dense_rank()

Duplicate values receive the same rank.

No gaps.

### Example

```text
Salary

100
100
90
```

Output:

```text
1
1
2
```

---

# lag()

Returns the previous row value.

```python
lag("salary", 1)
```

### Output

```text
50000 -> null
60000 -> 50000
70000 -> 60000
```

---

# lead()

Returns the next row value.

```python
lead("salary", 1)
```

### Output

```text
50000 -> 60000
60000 -> 70000
70000 -> null
```

---

## Common Window Functions

- row_number()
- rank()
- dense_rank()
- lag()
- lead()
- first()
- last()
- sum()
- avg()

---

## Interview Answer

**Window functions perform calculations across a set of related rows without aggregating them into a single row. Common window functions include row_number, rank, dense_rank, lag, and lead.**

---

# Quick Interview Revision

## Decorator

Adds extra functionality to a function without modifying the original code.

---

## Generator

Produces values lazily using `yield` and is memory efficient.

---

## Cache

Stores data in memory.

Default Storage Level:

```python
MEMORY_ONLY
```

---

## Persist

Stores data using a custom storage level.

---

## Catalyst Optimizer

Spark SQL query optimization engine.

---

## Tungsten Engine

Spark execution engine focused on memory and CPU optimization.

---

## AQE

Optimizes execution plans dynamically during runtime.

---

## Data Skew

Uneven data distribution across partitions.

---

## Salting

Technique used to solve data skew.

---

## Window Functions

Used for ranking, running totals, lag/lead, and row-wise analytics without collapsing rows.